# TP2 — Du Calcul Symbolique à l'IA Scientifique Multi-GPU
## Automatisation de la Reproductibilité (CI/CD)

**Formation Doctorale : Calcul Scientifique, HPC et Génie Logiciel Avancé — Cycle Ph.D. D1**

**Auteurs :**
- Mohamed Beddi Cheikh Ahmed — C16630
- Bouchra Dah — C10307

**Dépôt GitHub :** https://github.com/medbeddi/phd_integrator_project

---

Ce notebook exécute et illustre l'intégralité du pipeline scientifique implémenté dans
`src/` : dérivation symbolique, manipulation NumPy/Polars, analyse de stabilité,
accélération HPC (Numba), réseau de neurones informé par la physique (PINN, PyTorch) et
visualisation scientifique. Il sert de rapport de synthèse reproductible ; le détail
méthodologique complet (procédure de déploiement `uv`, analyse critique de précision,
CI/CD) est documenté dans le `README.md` du dépôt.


In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch

from src.symbolic_derivation import build_symbolic_context, derive_symbolic
from src.numerical_core import build_spatiotemporal_grid, describe_array, apply_residual_vectorized
from src.stability_analysis import analyze_stability, build_hilbert_matrix, condition_number
from src.hpc_acceleration import make_smoothing_kernel, benchmark_filter
from src.deep_pinn import PINN, get_device, predict, train_pinn
from src.visualization import plot_solution_and_error, plot_interactive_surface

from pathlib import Path
OUT = Path("../outputs")


## Module 3 — Dérivation symbolique (SymPy)

In [2]:
ctx = build_symbolic_context()
print("Solution candidate (onde solitaire):")
sp_display = ctx.u
print(f"  u(x,t) = {ctx.u}")
print(f"  du/dt  = {ctx.du_dt}")
print(f"  du/dx  = {ctx.du_dx}")
print(f"  d2u/dx2 = {ctx.d2u_dx2}")
print(f"  f(x,t) = {ctx.residual}   (terme source residuel exact)")


Solution candidate (onde solitaire):
  u(x,t) = -tanh(c*t - x)
  du/dt  = -c*(1 - tanh(c*t - x)**2)
  du/dx  = 1 - tanh(c*t - x)**2
  d2u/dx2 = -2*(tanh(c*t - x)**2 - 1)*tanh(c*t - x)
  f(x,t) = 2*nu*(tanh(c*t - x)**2 - 1)*tanh(c*t - x)   (terme source residuel exact)


In [3]:
functions = derive_symbolic()
u_num, f_num = functions["u"], functions["f"]
print("u(1.0, 0.0, c=1.0) =", u_num(1.0, 0.0, 1.0))
print("f(1.0, 0.0, c=1.0, nu=0.05) =", f_num(1.0, 0.0, 1.0, 0.05))


u(1.0, 0.0, c=1.0) = 0.7615941559557649
f(1.0, 0.0, c=1.0, nu=0.05) = 0.031985000422461225


## Module 4 — NumPy bas niveau, Polars et vectorisation

In [4]:
x_grid, t_grid = build_spatiotemporal_grid(n_space=100, m_time=100)
attrs = describe_array(x_grid)
print(attrs)

residual_grid = apply_residual_vectorized(f_num, x_grid, t_grid, c=1.0, nu=0.05)
print("Grille de residu vectorise:", residual_grid.shape, residual_grid.dtype)


GridAttributes(shape=(100, 100), dtype='float64', strides=(800, 8), c_contiguous=True, f_contiguous=False)
Grille de residu vectorise: (100, 100) float64


## Module 5 — Conditionnement et précision IEEE 754

In [5]:
reports = analyze_stability([5, 10, 15, 20, 25])
for r in reports:
    print(
        f"n={r.dimension:2d}  kappa={r.condition_number:.3e}  "
        f"err16={r.error_float16:.3e}  err32={r.error_float32:.3e}  err64={r.error_float64:.3e}"
    )


n= 5  kappa=4.766e+05  err16=7.441e-04  err32=7.484e-08  err64=2.719e-16
n=10  kappa=1.602e+13  err16=1.727e-03  err32=2.181e-07  err64=5.661e-16
n=15  kappa=3.676e+17  err16=1.639e-03  err32=1.640e-07  err64=1.211e-15
n=20  kappa=1.319e+18  err16=5.735e-03  err32=5.279e-07  err64=1.434e-14
n=25  kappa=1.331e+18  err16=1.042e-03  err32=8.812e-07  err64=1.794e-15


In [6]:
import matplotlib.pyplot as plt

dims = [r.dimension for r in analyze_stability()]
kappas = [r.condition_number for r in analyze_stability()]

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(dims, kappas, marker="o")
ax.set_xlabel("Dimension n de la matrice de Hilbert")
ax.set_ylabel("Nombre de conditionnement kappa(A)")
ax.set_title("Croissance du conditionnement (echelle log)")
plt.tight_layout()
plt.show()


C:\Users\beddi\AppData\Local\Temp\ipykernel_28132\4204123991.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Module 6 — Accélération HPC (Numba JIT)

In [7]:
rng = np.random.default_rng(0)
grid = rng.standard_normal((300, 300))
kernel = make_smoothing_kernel(5)

result = benchmark_filter(grid, kernel, repeats=2)
print(f"Python pur : {result.python_seconds*1000:.2f} ms")
print(f"Numba JIT  : {result.numba_seconds*1000:.2f} ms")
print(f"Speedup    : x{result.speedup:.1f}")


Python pur : 1277.65 ms
Numba JIT  : 20.67 ms
Speedup    : x61.8


## Module 7 — Physics-Informed Neural Network (PyTorch)

In [8]:
torch.manual_seed(0)
model = PINN(hidden_size=32, n_hidden_layers=4)

x_c = torch.linspace(-5, 5, 40).reshape(-1, 1)
t_c = torch.linspace(0, 2, 40).reshape(-1, 1)
t_b = torch.linspace(0, 2, 20).reshape(-1, 1)
x_b = torch.cat([torch.full_like(t_b, -5.0), torch.full_like(t_b, 5.0)])
t_b = torch.cat([t_b, t_b])
u_b = torch.tanh(x_b - 1.0 * t_b)
x_s = torch.linspace(-5, 5, 20).reshape(-1, 1)
t_s = torch.zeros((20, 1))
u_s = torch.tanh(x_s)

history = train_pinn(
    model, (x_c, t_c), (x_b, t_b), u_b, (x_s, t_s), u_s,
    c=1.0, nu=0.05, epochs=150, lr=1e-3,
)
print(f"Device: {get_device()}")
print(f"Perte totale — initiale: {history.total[0]:.4e}, finale: {history.total[-1]:.4e}")


Device: cpu
Perte totale — initiale: 1.7162e+00, finale: 1.0141e-02


In [9]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history.total, label="totale")
ax.plot(history.physics, label="physique", alpha=0.7)
ax.plot(history.boundary, label="conditions aux limites", alpha=0.7)
ax.plot(history.data, label="donnees capteurs", alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("Epoque")
ax.set_ylabel("Perte (echelle log)")
ax.set_title("Convergence de l'entrainement du PINN")
ax.legend()
plt.tight_layout()
plt.show()


C:\Users\beddi\AppData\Local\Temp\ipykernel_28132\820212787.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Module 8 — Visualisation scientifique

In [10]:
x_flat = torch.tensor(x_grid.reshape(-1, 1), dtype=torch.float32)
t_flat = torch.tensor(t_grid.reshape(-1, 1), dtype=torch.float32)
u_predicted = predict(model, x_flat, t_flat).numpy().reshape(x_grid.shape)
u_reference = torch.tanh(x_flat - 1.0 * t_flat).numpy().reshape(x_grid.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
mesh = ax1.pcolormesh(t_grid, x_grid, u_predicted, shading="auto", cmap="viridis")
ax1.set_xlabel("t"); ax1.set_ylabel("x"); ax1.set_title("Solution approchee (PINN)")
fig.colorbar(mesh, ax=ax1)

error = np.abs(u_predicted - u_reference)
ax2.plot(t_grid[0, :], error.mean(axis=0), color="firebrick")
ax2.set_xlabel("t"); ax2.set_ylabel("Erreur absolue moyenne"); ax2.set_title("Evolution de l'erreur")
plt.tight_layout()
plt.show()


C:\Users\beddi\AppData\Local\Temp\ipykernel_28132\1822939390.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusion

Ce notebook confirme, de bout en bout, le bon fonctionnement du pipeline reproductible :
dérivation symbolique exacte du terme source (méthode des solutions manufacturées),
vectorisation NumPy/Polars sans boucle explicite, quantification de la sensibilité
numérique via le conditionnement de la matrice de Hilbert, accélération JIT mesurable via
Numba, et convergence de la perte du PINN vers une approximation cohérente avec la
solution analytique `u(x,t) = tanh(x - c·t)`. L'ensemble de ces étapes est automatisé et
validé en continu par le pipeline CI/CD GitHub Actions décrit dans le `README.md`.

**Mohamed Beddi Cheikh Ahmed (C16630) — Bouchra Dah (C10307)**
